# 02 · DIFF vs BOUNDARY  —  is the leftover gated by a completion clock `c`, and is `boundary = DIFF==0`?

- **D1** reference readout of the DIFF==0 / all-done state — what does "nothing left" verbalize as?
- **D2** backup-goal-heads — do BOUNDARY heads also read out the frontier?
- **D4** done-state transplant — inject the all-done goal-write into a mid-task pivot: does it terminate? (`DIFF==0 ⇒ boundary`)
- **D3** competition vs upstream *(PRELIMINARY)* — boundary-done injection suppresses the frontier; does boosting the goal write rescue it?

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2, 3, 4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("..")  # so `_diff_primitives` is importable
import torch, numpy as np
import _diff_primitives as P

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")  # loads model + head masks (once)
from _diff_primitives import *  # helper functions into scope

mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = (
    P.GOAL_HEADS,
    P.BOUNDARY_HEADS,
    P.UNION_HEADS,
    P.GROUP,
)

2026-08-31 17:21:45 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [20]:
# ── multipredicate example: 4 INDEPENDENT computations, one assignment line each ──
# All orderings valid — the pivot choice is a genuine "which sub-task next", not a data dependency.
# Item `a` is the ANCHOR: ALWAYS prefilled — one in-context example that fixes the coding style
# (an inline list-comprehension). Progress analysis focuses on the remaining {b, c, d}.

USE_NATURAL_NAMES = False        # False → slot names a/b/c/d ; True → descriptive variable names
ITEMS_CONFIG      = "words_B"    # ← pick a preset from ITEMS_PRESETS below

# each preset: slot -> (condition text, RHS expression, natural variable name).
# convention: slot `a` is always a list-comprehension filter (the anchor / style example).
# all presets share the domain  def func(words: list[str])  so build_prefill/QUESTION stay valid.
ITEMS_PRESETS = {
    "words_A": {
        "a": ("the list of words that start with 'a'",          "[w for w in words if w.startswith('a')]",           "a_words"),
        "b": ("the count of palindromic words",                 "sum(w == w[::-1] for w in words)",                  "palindrome_count"),
        "c": ("the length of the longest word",                 "max(len(w) for w in words)",                        "longest_len"),
        "d": ("the list of words with more than five letters",  "[w for w in words if len(w) > 5]",                  "long_words"),
    },
    "words_B": {
        "a": ("the list of words that contain the letter 'e'",  "[w for w in words if 'e' in w]",                    "words_with_e"),
        "b": ("the total number of characters across all words","sum(len(w) for w in words)",                        "total_chars"),
        "c": ("the number of distinct words",                   "len(set(words))",                                   "distinct_count"),
        "d": ("the list of words sorted by length",             "sorted(words, key=len)",                            "by_length"),
    },
    "words_C": {
        "a": ("the list of words that end with 's'",            "[w for w in words if w.endswith('s')]",             "plural_words"),
        "b": ("the average word length",                        "sum(len(w) for w in words) / len(words)",           "avg_len"),
        "c": ("the number of words containing a digit",         "sum(any(ch.isdigit() for ch in w) for w in words)", "digit_word_count"),
        "d": ("the list words in words that are palindromes",   "[w for w in words if w == w[::-1]]",                   "palindromes"),
    },
}
ITEMS   = ITEMS_PRESETS[ITEMS_CONFIG]
ORDER   = ["a", "b", "c", "d"]
ANCHOR  = "a"                    # always prefilled, never a pivot choice
ANALYZE = ["b", "c", "d"]       # the progress set we sweep / inject over

def varname(k): return ITEMS[k][2] if USE_NATURAL_NAMES else k
def assign(k):  return f"{varname(k)} = {ITEMS[k][1]}"

# conditions only — no dictated variable names; the LM names what it writes at/after the pivot
TASK_SPEC = "\n".join(f"({i+1}) {ITEMS[k][0]}" for i, k in enumerate(ORDER))
QUESTION = ("Write a Python program that implements a function `func` which, given a list of "
            "strings `words`, computes:\n" + TASK_SPEC + "\n")

def build_prefill(done, extra=""):
    """Prefill = ANCHOR (always) + assignments in `done` (⊆ ANALYZE), kept in ORDER; pivot at next indent."""
    keep = [k for k in ORDER if k == ANCHOR or k in done]
    lines = ["<code>", "def func(words: list[str]):"] + [f"    {assign(k)}" for k in keep]
    return "\n".join(lines) + "\n    " + extra   # trailing indent = pivot (about to write next var)

def build_prompt(done, extra=""):
    return get_prompt(QUESTION, build_prefill(done, extra))

def pivot_of(prompt):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] - 1

# task-item token spans are identical across done-sets (task precedes prefill) — compute once.
_p0 = build_prompt([])                        # {} means "anchor a only"
ITEM_SPAN = {k: token_span(_p0, ITEMS[k][0]) for k in ORDER}
show_prompt("done={a} (anchor only)", _p0,               ITEM_SPAN["b"])
show_prompt("done={a,b}           ", build_prompt(["b"]), ITEM_SPAN["c"])
print(f"config={ITEMS_CONFIG}  item spans:", ITEM_SPAN, " pivot({a})=", pivot_token(_p0))

╭─ done={a} (anchor only)  TASK span (80, 88)  ·  pivot=['    ']  ·  L=141 ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that contain the letter 'e'\n                                                             │
│ (2) the total number of characters across all words\n                                                           │
│ (3) the number of distinct words\n                                                                              │
│ (4) the list of words sorted by length\n                                                                        │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if 'e' in w]\n                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ done={a,b}             TASK span (92, 97)  ·  pivot=['    ']  ·  L=153 ────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that contain the letter 'e'\n                                                             │
│ (2) the total number of characters across all words\n                                                           │
│ (3) the number of distinct words\n                                                                              │
│ (4) the list of words sorted by length\n                                                                        │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if 'e' in w]\n                                                                        │
│     b = sum(len(w) for w in words)\n                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

config=words_B  item spans: {'a': (66, 77), 'b': (80, 88), 'c': (92, 97), 'd': (101, 108)}  pivot({a})= ['    ']


In [3]:
# canonical mid-task instance: anchor {a} done, frontier = b ; and the all-done (DIFF==0) instance.
p_mid  = build_prompt([]);                               piv_mid  = pivot_of(p_mid)   # anchor {a}; frontier b
p_done = build_prompt(["b", "c", "d"], extra="return");  piv_done = pivot_of(p_done)  # all done (DIFF==0)
show_prompt("MID (frontier=b)", p_mid,  ITEM_SPAN["b"])
show_prompt("DONE (DIFF==0)  ", p_done, ITEM_SPAN["d"])
print("pivots: mid=", pivot_token(p_mid), " done=", pivot_token(p_done))

╭─ MID (frontier=b)  TASK span (79, 83)  ·  pivot=['    ']  ·  L=141 ─────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list words in words that are palindromes\n                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DONE (DIFF==0)    TASK span (98, 109)  ·  pivot=[' return']  ·  L=192 ─────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list words in words that are palindromes\n                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = sum(len(w) for w in words) / len(words)\n                                                               │
│     c = sum(any(ch.isdigit() for ch in w) for w in words)\n                                                     │
│     d = [w for w in words if w == w[::-1]]\n                                                                    │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

pivots: mid= ['    ']  done= [' return']


## D1 · Reference readout of the DIFF==0 (all-done) state

What does the goal-head readout say when nothing is left? This is the reference for D3/D4 —
"boundary" behaviour. **Predict:** "return / done / nothing left", not a task item.

In [4]:
# D1: readout the goal-head write at the all-done (DIFF==0) pivot. No intervention — pure readout.
show_prompt(f"D1 · DONE (DIFF==0) · readout @ pivot idx {piv_done}", p_done, ITEM_SPAN["d"])
goalscope_readout("DONE (DIFF==0) ", p_done, piv_done, heads=GOAL_HEADS, maxnew=64)
rich_print(f"[grey70]clean DONE next-gen (expect return):[/grey70] "
           f"{escape(repr(greedy(prepare_input(prompts=p_done, tokenizer=mt), max_new=24)))}")

╭─ D1 · DONE (DIFF==0) · readout @ pivot idx 191  TASK span (98, 109)  ·  pivot=[' return']  ·  L=192 ────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list words in words that are palindromes\n                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│     b = sum(len(w) for w in words) / len(words)\n                                                               │
│     c = sum(any(ch.isdigit() for ch in w) for w in words)\n                                                     │
│     d = [w for w in words if w == w[::-1]]\n                                                                    │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


DONE (DIFF==0)   pos=191 ' return' => Sure, please provide the specific line or lines of code you're referring to.

clean DONE next-gen (expect return): ' a, b, c, d\n</code>'

## D2 · Backup-goal-heads check

Read the frontier off the **BOUNDARY** heads instead of the GOAL heads (same mid-task pivot). If they
verbalize the frontier ≈ as well ⇒ redundant carriers (leans H2; predicts goal-head ablation won't kill
DIFF). If they don't ⇒ BOUNDARY_PATCH acts by same-layer competition, not by carrying DIFF.

In [5]:
# D2: read the SAME mid-task pivot off GOAL vs BOUNDARY vs UNION heads. No intervention — readout only.
show_prompt(f"D2 · MID (frontier=b) · readout @ pivot idx {piv_mid}", p_mid, ITEM_SPAN["b"])
goalscope_readout("GOAL heads     ", p_mid, piv_mid, heads=GOAL_HEADS,     maxnew=48)
goalscope_readout("BOUNDARY heads ", p_mid, piv_mid, heads=BOUNDARY_HEADS, maxnew=48)
goalscope_readout("UNION heads    ", p_mid, piv_mid, heads=UNION_HEADS,    maxnew=48)

╭─ D2 · MID (frontier=b) · readout @ pivot idx 140  TASK span (79, 83)  ·  pivot=['    ']  ·  L=141 ──────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that end with 's'\n                                                                       │
│ (2) the average word length\n                                                                                   │
│ (3) the number of words containing a digit\n                                                                    │
│ (4) the list words in words that are palindromes\n                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = [w for w in words if w.endswith('s')]\n                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GOAL heads       pos=140 '    ' => Calculating the average length of words in a string.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

BOUNDARY heads   pos=140 '    ' => I'm analyzing the current line of code to provide a concise explanation.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

UNION heads      pos=140 '    ' => I'm calculating the average length of words in a string.

[{'pos': 140,
  'token': '    ',
  'patched_gen': "I'm calculating the average length of words in a string.",
  'scoped': True}]

## D4 · Move-on triggers — boundary move-on (clock/16) vs goal DIFF==0

Replaces the old done-state transplant, which conflated *local* ("this predicate is done → next var")
with *global* ("all done → return") boundaries. On the **multi-line multi-predicate** setup
(`build_example`, int predicates — exactly the clock/16 move-on domain), compare two causal triggers
at the pivot, both via `headpatched_generate` (replace the heads' o_proj input, `n_steer=1`, `alpha=2`):

- **(i) boundary move-on** — pooled move-on vector over the **decision/BOUNDARY heads**, exactly
  `clock/16_move_on_signal` (`MEAN_SRC` = mean single-line-style o_proj @ `PATCH_POS`, patched over `DEC_MASK`).
- **(ii) goal DIFF==0** — the **same** single-line "completed" source contexts, read off the **GOAL heads**
  and patched over `GOAL_MASK` (apples-to-apples with (i)).

**Metric:** does the mid-task run write the **next variable** (move-on ✓)? Both ⇒ goal-DIFF==0 and
boundary-move-on carry overlapping "advance" signals; only (i) ⇒ move-on lives in `c`.

In [6]:
# ── D4 setup: clock/16 move-on machinery (decision heads, PATCH_POS, pooled move-on / DIFF==0 vecs) ──
# Affinity is NOT recomputed here: `decision_heads = BOUNDARY_HEADS` already comes from clock/15's
# cached aff_sep (loaded by _diff_primitives.setup()). Only MEAN_SRC/DIFF0_GOAL are computed, and they
# are disk-cached (clock/15's `cached()` pattern) -> first run is slow (~minutes), later runs load.
import random, os, json, hashlib
from pathlib import Path
from src.utils import env_utils
from src.goals.oid_gaze import cache_root
from src.goals.head_loc_attn import HeadMask, get_o_proj_inputs_batch
from src.goals.completion_pairs import build_completion_pairs
from src.goals.multiline_prompts import build_example, replace_in_code_newline_with_tabnewline
from src.goals.goalscope import headpatched_generate

MODEL_NAME, DATATYPE = "Qwen/Qwen2.5-Coder-32B-Instruct", "int"   # move-on mask+vector derived on int
_root = os.path.join(cache_root(MODEL_NAME, "conditions"), "completion_mask")
_latest = max([t for t in os.listdir(_root) if os.path.isdir(os.path.join(_root, t)) and "lam10.0" in t],
              key=lambda t: os.path.getmtime(os.path.join(_root, t)))
DCK = torch.load(os.path.join(_root, _latest, "mask.pt"), map_location="cpu", weights_only=False)
dec_layers = [int(x) for x in DCK["layers"]]
PATCH_POS  = int(DCK["patch_tokens"][0]) if len(DCK.get("patch_tokens", [-2])) == 1 else -2
ALPHA = 2.0
decision_heads = BOUNDARY_HEADS      # clock/16's affinity-filtered decision heads (clock/15 aff_sep cache)

def get_mask_bin(heads, layers):
    # rows align POSITIONALLY to `layers` (row li <-> layers[li]) — forward_with_src_replacement
    # requires len(mask_layers) == src layer axis == mask rows, so the mask can't be full-depth.
    idx = {l: i for i, l in enumerate(layers)}
    hm = HeadMask(len(layers), mt.model.config.num_attention_heads)
    with torch.no_grad():
        hm.values.zero_()
        for l, h in heads:
            if l in idx: hm.values[idx[l], h] = 1.0
    return hm
GOAL_LAYERS = sorted({l for l, _ in GOAL_HEADS})
DEC_MASK   = get_mask_bin(decision_heads, dec_layers)
GOAL_MASK_ = get_mask_bin(GOAL_HEADS, GOAL_LAYERS)
print(f"decision heads {len(decision_heads)} · PATCH_POS={PATCH_POS} · goal heads {len(GOAL_HEADS)} "
      f"(layers {GOAL_LAYERS[0]}-{GOAL_LAYERS[-1]})")

# ── disk cache (clock/15's cached()): MEAN_SRC/DIFF0_GOAL are a pure function of this config ──
RES = env_utils.DEFAULT_RESULTS_DIR
CACHE_DIR = Path(RES) / "notebook_caches" / "02_diff_vs_boundary"
CACHE_OVERWRITE = False               # True -> recompute + rewrite
def _cache_key(d): return hashlib.md5(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:10]
def cached(name, key, build, overwrite=None):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    fp = CACHE_DIR / f"{name}__{_cache_key(key)}.pt"
    if fp.exists() and not (CACHE_OVERWRITE if overwrite is None else overwrite):
        print(f"  [cache hit ] {name} ({fp.name})")
        return torch.load(fp, map_location="cpu", weights_only=False)["value"]
    val = build(); torch.save({"value": val, "key": key}, fp)
    print(f"  [cache miss] {name} -> wrote {fp.name}")
    return val

SLOTS, N_SRC, SRC_BATCH = [1, 2, 3], 128, 16
def _mean_at_pivot(prompts, layers):
    acc = {l: [] for l in layers}
    for i in range(0, len(prompts), SRC_BATCH):
        c = get_o_proj_inputs_batch(mt, prompts[i:i+SRC_BATCH], layers, token_positions=[PATCH_POS])
        for l in layers: acc[l].append(c[l][:, 0, :].float().cpu())
    return torch.stack([torch.cat(acc[l], 0).mean(0) for l in layers])

def _build_vecs():
    _sl = []
    for s in SLOTS:
        pairs = build_completion_pairs(mt, datatype=DATATYPE, n_pairs=N_SRC, target_slots=(s,),
                                       direction="sl->ml", same=True, patch_tokens=(PATCH_POS,),
                                       seed=1234 + s, log=lambda *a, **k: None)
        _sl += [pr.source.sl_prompt for pr in pairs]
    return {"move_on_dec": _mean_at_pivot(_sl, dec_layers),      # (n_dec_layers,  n_embd) — (i)
            "diff0_goal":  _mean_at_pivot(_sl, GOAL_LAYERS),     # (n_goal_layers, n_embd) — (ii)
            "n_ctx": len(_sl)}

_key = dict(model=MODEL_NAME, datatype=DATATYPE, slots=SLOTS, n_src=N_SRC, patch_pos=PATCH_POS,
            dec_layers=dec_layers, goal_layers=GOAL_LAYERS)
_vecs = cached("moveon_diff0_vecs", _key, _build_vecs)
MOVE_ON_DEC, DIFF0_GOAL = _vecs["move_on_dec"], _vecs["diff0_goal"]
print(f"contexts={_vecs['n_ctx']} · MOVE_ON_DEC {tuple(MOVE_ON_DEC.shape)} · DIFF0_GOAL {tuple(DIFF0_GOAL.shape)}")

decision heads 140 · PATCH_POS=-2 · goal heads 122 (layers 28-55)
  [cache hit ] moveon_diff0_vecs (moveon_diff0_vecs__e5209fff5a.pt)
contexts=384 · MOVE_ON_DEC (64, 5120) · DIFF0_GOAL (26, 5120)


In [25]:
# ── D4 dest: a multi-line multi-predicate function, pivoted mid-implementation of a slot ──
def build_moveon_dest(rng, slot=2, n_cond=5, naming="letters"):
    ex = build_example(mt=mt, rng=rng, target_slot=slot, n_conditions=n_cond, mode="chat",
                       style="multiline", naming=naming, datatype=DATATYPE)
    # return replace_in_code_newline_with_tabnewline(ex["full_prompt"]) + "   ", ex
    return ex["full_prompt"] + "   ", ex

d4_prompt, d4_ex = build_moveon_dest(random.Random(7), slot=1, n_cond=5, naming="natural")
rich_print(Panel(Text.from_markup(exact_prompt(d4_prompt, patch_pos=[PATCH_POS])),
                 title=f"[bold]D4 DEST · multi-line · pivot(PATCH_POS={PATCH_POS}) in salmon · "
                       f"var_names={d4_ex['var_names']}[/bold]",
                 title_align="left", border_style="grey42", padding=(1, 1)))

╭─ D4 DEST · multi-line · pivot(PATCH_POS=-2) in salmon · var_names=['is_power_of_two', 'is_perfect_cube', 'has_l─╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical   │
│ correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>\n                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `check_conditions` which, given an integer n, checks the      │
│ following conditions:\n                                                                                         │
│ \n                                                                                                              │
│ 1. n is a power of two\n                                                                                        │
│ 2. n is a perfect cube\n                                                                                        │
│ 3. the sum of n's digits is greater than 5\n                                                                    │
│ 4. n is odd\n                                                                                                   │
│ 5. all of n's digits are even\n                                                                                 │
│ \n                                                                                                              │
│ For each condition, in the order listed, assign the boolean result of the check to its own local variable.      │
│ After all conditions have been checked, use a SINGLE print statement that, using those variables, reports       │
│ whether each condition passed individually and whether all of them passed collectively.\n                       │
│ Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not    │
│ define any helper functions.\n                                                                                  │
│ Do not include any comments in your code.<|im_end|>\n                                                           │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def check_conditions(n: int):\n                                                                                 │
│     is_power_of_two = bin(n).count("1")\n                                                                       │
│     is_power_of_two = n > 0 and is_power_of_two == 1\n                                                          │
│     is_perfect_cube = round(abs(n) ** (1 / 3))\n                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [26]:
# ── D4 interventions: clean vs (i) boundary move-on vs (ii) goal DIFF==0, patched @ PATCH_POS ──
d4_inp = prepare_input(prompts=d4_prompt, tokenizer=mt)
clean  = headpatched_generate(mt, dict(d4_inp), MOVE_ON_DEC, get_mask_bin([], dec_layers), dec_layers,
                              max_new_tokens=24, patch_input_positions=[PATCH_POS])          # empty mask = pure cascade
moveon = headpatched_generate(mt, dict(d4_inp), MOVE_ON_DEC, DEC_MASK, dec_layers, n_steer_tokens=1,
                              patch_input_positions=[PATCH_POS], alpha=ALPHA)                # (i)
diff0  = headpatched_generate(mt, dict(d4_inp), DIFF0_GOAL, GOAL_MASK_, GOAL_LAYERS, n_steer_tokens=1,
                              patch_input_positions=[PATCH_POS], alpha=ALPHA)                # (ii)

# (iii) UNION DIFF==0 — patch goal ∪ boundary heads with the completed-context write. The o_proj
# input at a layer is head-agnostic, so DIFF0_GOAL / MOVE_ON_DEC are identical on shared layers;
# just merge them over UNION_LAYERS (no recompute) and patch the union head mask.
UNION_LAYERS = sorted({l for l, _ in UNION_HEADS})
_g = {l: DIFF0_GOAL[i] for i, l in enumerate(GOAL_LAYERS)}
_d = {l: MOVE_ON_DEC[i] for i, l in enumerate(dec_layers)}
DIFF0_UNION = torch.stack([_g[l] if l in _g else _d[l] for l in UNION_LAYERS])   # (n_union_layers, n_embd)
UNION_MASK  = get_mask_bin(UNION_HEADS, UNION_LAYERS)
uni    = headpatched_generate(mt, dict(d4_inp), DIFF0_UNION, UNION_MASK, UNION_LAYERS, n_steer_tokens=1,
                              patch_input_positions=[PATCH_POS], alpha=ALPHA)                # (iii)
rich_print(f"[grey70]clean                   :[/grey70] {escape(repr(clean))}")
rich_print(f"[sea_green3](i)   boundary move-on   :[/sea_green3] {escape(repr(moveon))}")
rich_print(f"[gold3](ii)  goal DIFF==0       :[/gold3] {escape(repr(diff0))}")
rich_print(f"[cyan](iii) union DIFF==0      :[/cyan] {escape(repr(uni))}")

clean                   : ' is_perfect_cube = is_perfect_cube ** 3 == n\n    sum_of_digits = sum(int(digit'

(i)   boundary move-on   : ' sum_of_digits = sum(int(digit) for digit in str(n))\n    
is_sum_of_digits_greater_than_five = sum_of_digits > 5\n    is_odd = n % 2 != 0\n    all_digits_even = 
all(int(digit) % 2 == 0 for digit in str(n'

(ii)  goal DIFF==0       : ' is_perfect_cube = is_perfect_cube ** 3 == n\n    sum_of_digits = sum(int(digit) for 
digit in str(n))\n    sum_of_digits_greater_than_five = sum_of_digits > 5\n    is_odd = n % 2 != 0\n    
all_digits_even = all'

(iii) union DIFF==0      : ' sum_of_digits = sum(int(digit) for digit in str(n))\n    
is_sum_of_digits_greater_than_five = sum_of_digits > 5\n    is_odd = n % 2 != 0\n    all_digits_even = 
all(int(digit) % 2 == 0 for digit in str(n'

### D4-rev · move-on / DIFF==0 on REVERSE-execution-order prompts (qualitative)

The TASK lists conditions 1..n, but a `REVERSE_CLAUSE` instructs implementing the **LAST first**. The prefill emits conditions last→target (descending) and pivots mid-`target`, so **next-EXECUTED** = `target−1` (the previous-*listed*) while **next-LISTED** = `target+1`. If the move-on / DIFF==0 signal drives *execution*-order transitions, the patched run should head to `target−1`, not `target+1`. Ported from `clock/16_move_on_signal`. Goal-head DIFF==0 uses a higher α (=6) than boundary move-on (α=2), as observed on the forward example.

In [29]:
# ── D4-rev: move-on / DIFF==0 on REVERSE-execution-order prompts (ref clock/16) ──
from src.goals.condition_data import INT_PREDICATES, ConditionSample, ConditionSpec, sample_test_inputs
from src.goals.functional import apply_chat_template

ALPHA_GOAL = 6.0   # goal-head DIFF==0 needs a higher coeff than boundary move-on (α=ALPHA≈2)
REVERSE_CLAUSE = ("The conditions should be checked in the REVERSE order of how they are mentioned "
                  "in this prompt: implement the LAST condition first and the FIRST condition last.")

def build_reverse_example(rng, target_slot, n_cond=5):
    """Prefill implements conditions last-to-first; pivot mid-`target_slot`.
    next-EXECUTED = target_slot-1 (prev-listed) ; next-LISTED = target_slot+1."""
    assert 1 <= target_slot <= n_cond - 2, "need >=1 executed prior (slot>target) and a successor"
    pool = [p for p in INT_PREDICATES if p.is_multiline_for("python")]
    preds = rng.sample(pool, n_cond)
    params = [p.sample_params(rng) for p in preds]
    var_names = [p.var_name for p in preds]
    spec = ConditionSpec(
        spec_id=f"rev_s{target_slot}_{rng.randrange(10**6):06d}",
        predicate_ids=[p.id for p in preds], predicate_params=params,
        spec_texts=[p.render(pr) for p, pr in zip(preds, params)],
        id_substrs=[p.render_substrs(pr) for p, pr in zip(preds, params)],
        test_inputs=sample_test_inputs(rng, "int"))
    sample = ConditionSample(spec=spec, lang="python", modularity="monolithic", without_comment=True)
    prior_slots = list(range(n_cond - 1, target_slot, -1))                # slots ABOVE target, descending
    styles = [rng.choice(["multiline", "singleline"]) for _ in prior_slots]
    if "multiline" not in styles: styles[rng.randrange(len(styles))] = "multiline"
    body = []
    for j, st in zip(prior_slots, styles):
        pp, prm, v = preds[j], params[j], var_names[j]
        body += ([pp.render_singleline(prm, v, lang="python")]
                 if st == "singleline" and pp.has_singleline_for("python")
                 else pp.render_multiline(prm, v, lang="python"))
    tp, tprm, tvar = preds[target_slot], params[target_slot], var_names[target_slot]
    body += [tp.render_multiline(tprm, tvar, lang="python")[0]]           # incomplete -> pivot
    instr = sample.render_instruction(name_rule="free").strip() + "\n\n" + REVERSE_CLAUSE
    full = apply_chat_template(mt=mt, user_query=instr,
                               incomplete_code=sample.render_code_stub() + "\n".join(body) + "\n")
    return dict(prompt=replace_in_code_newline_with_tabnewline(full) + "   ",
                names=var_names, cur=target_slot, nxt_exec=target_slot - 1, nxt_listed=target_slot + 1)

def d4_arms(prompt, max_new=26):
    inp = prepare_input(prompts=prompt, tokenizer=mt)
    clean  = headpatched_generate(mt, dict(inp), MOVE_ON_DEC, get_mask_bin([], dec_layers), dec_layers,
                                  max_new_tokens=max_new, patch_input_positions=[PATCH_POS])
    moveon = headpatched_generate(mt, dict(inp), MOVE_ON_DEC, DEC_MASK, dec_layers, n_steer_tokens=1,
                                  max_new_tokens=max_new, patch_input_positions=[PATCH_POS], alpha=ALPHA)
    diff0  = headpatched_generate(mt, dict(inp), DIFF0_GOAL, GOAL_MASK_, GOAL_LAYERS, n_steer_tokens=1,
                                  max_new_tokens=max_new, patch_input_positions=[PATCH_POS], alpha=ALPHA_GOAL)
    uni    = headpatched_generate(mt, dict(inp), DIFF0_UNION, UNION_MASK, UNION_LAYERS, n_steer_tokens=1,
                                  max_new_tokens=max_new, patch_input_positions=[PATCH_POS], alpha=ALPHA_GOAL)
    return clean, moveon, diff0, uni

for _i, _slot in enumerate([2, 1]):                                        # 1-2 fixed reverse prompts
    inst = build_reverse_example(random.Random(27 + _i), target_slot=_slot, n_cond=5)
    nm = inst["names"]
    rich_print(Panel(Text.from_markup(exact_prompt(inst["prompt"], patch_pos=[PATCH_POS])),
                     title=f"[bold]D4-REV #{_i} · current={nm[inst['cur']]!r} · "
                           f"next-EXEC={nm[inst['nxt_exec']]!r} · next-LISTED={nm[inst['nxt_listed']]!r} "
                           f"· pivot@{PATCH_POS}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))
    clean, moveon, diff0, uni = d4_arms(inst["prompt"])
    rich_print(f"[grey70]clean                       :[/grey70] {escape(repr(clean))}")
    rich_print(f"[sea_green3](i)   boundary move-on α={ALPHA}  :[/sea_green3] {escape(repr(moveon))}")
    rich_print(f"[gold3](ii)  goal DIFF==0 α={ALPHA_GOAL}    :[/gold3] {escape(repr(diff0))}")
    rich_print(f"[cyan](iii) union DIFF==0 α={ALPHA_GOAL}   :[/cyan] {escape(repr(uni))}")

╭─ D4-REV #0 · current='has_many_set_bits' · next-EXEC='has_min_digits' · next-LISTED='is_palindrome' · pivot@-2 ─╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical   │
│ correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>\n                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `check_conditions` which, given an integer n, checks the      │
│ following conditions:\n                                                                                         │
│ \n                                                                                                              │
│ 1. all of n's digits are even\n                                                                                 │
│ 2. n has at least 2 digits\n                                                                                    │
│ 3. n has more than 3 ones in its binary representation\n                                                        │
│ 4. n is a palindrome (its digits read the same forwards and backwards)\n                                        │
│ 5. n is odd\n                                                                                                   │
│ \n                                                                                                              │
│ For each condition, in the order listed, assign the boolean result of the check to its own local variable.      │
│ After all conditions have been checked, use a SINGLE print statement that, using those variables, reports       │
│ whether each condition passed individually and whether all of them passed collectively.\n                       │
│ Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not    │
│ define any helper functions.\n                                                                                  │
│ Do not include any comments in your code.\n                                                                     │
│ \n                                                                                                              │
│ The conditions should be checked in the REVERSE order of how they are mentioned in this prompt: implement the   │
│ LAST condition first and the FIRST condition last.<|im_end|>\n                                                  │
│ <|im_start|>assistant\n                                                                                         │
│ <code>    \n                                                                                                    │
│ def check_conditions(n: int):    \n                                                                             │
│     is_odd = n % 2 == 1    \n                                                                                   │
│     is_palindrome = str(abs(n))    \n                                                                           │
│     is_palindrome = is_palindrome == is_palindrome[::-1]    \n                                                  │
│     has_many_set_bits = bin(abs(n)).count("1")    \n                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean                       : ' has_many_set_bits = has_many_set_bits > 3    \n    has_at_least_two_digits = 
len(str(abs(n))) >='

(i)   boundary move-on α=2.0  : ' has_at_least_two_digits = len(str(abs(n))) >= 2    \n    all_even_digits = 
all(int(digit) %'

(ii)  goal DIFF==0 α=6.0    : ' has_at_least_two_digits = len(str(abs(n))) >= 2    \n    all_even_digits = 
all(int(digit) %'

(iii) union DIFF==0 α=6.0   : ' has_at_least_two_digits = len(str(abs(n))) >= 2    \n    all_even_digits = 
all(int(digit) %'

╭─ D4-REV #1 · current='is_perfect_cube' · next-EXEC='is_perfect_square' · next-LISTED='is_triangular' · pivot@-2─╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical   │
│ correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>\n                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `check_conditions` which, given an integer n, checks the      │
│ following conditions:\n                                                                                         │
│ \n                                                                                                              │
│ 1. n is a perfect square\n                                                                                      │
│ 2. n is a perfect cube\n                                                                                        │
│ 3. n is a triangular number\n                                                                                   │
│ 4. n is smaller than the number obtained by reversing its digits\n                                              │
│ 5. n is divisible by 16\n                                                                                       │
│ \n                                                                                                              │
│ For each condition, in the order listed, assign the boolean result of the check to its own local variable.      │
│ After all conditions have been checked, use a SINGLE print statement that, using those variables, reports       │
│ whether each condition passed individually and whether all of them passed collectively.\n                       │
│ Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not    │
│ define any helper functions.\n                                                                                  │
│ Do not include any comments in your code.\n                                                                     │
│ \n                                                                                                              │
│ The conditions should be checked in the REVERSE order of how they are mentioned in this prompt: implement the   │
│ LAST condition first and the FIRST condition last.<|im_end|>\n                                                  │
│ <|im_start|>assistant\n                                                                                         │
│ <code>    \n                                                                                                    │
│ def check_conditions(n: int):    \n                                                                             │
│     is_divisible = n % 16    \n                                                                                 │
│     is_divisible = is_divisible == 0    \n                                                                      │
│     is_reverse_greater = int(str(abs(n))[::-1])    \n                                                           │
│     is_reverse_greater = is_reverse_greater > abs(n)    \n                                                      │
│     is_triangular = int((2 * abs(n)) ** 0.5)    \n                                                              │
│     is_triangular = n >= 0 and is_triangular * (is_triangular + 1) // 2 == n    \n                              │
│     is_perfect_cube = round(abs(n) ** (1 / 3))    \n                                                            │
│                                                       

clean                       : ' is_perfect_cube = is_perfect_cube ** 3 == abs(n)    \n    is_perfect_square = 
round(abs(n'

(i)   boundary move-on α=2.0  : ' is_perfect_square = int(abs(n) ** 0.5)    \n    is_perfect_square = 
is_perfect_square'

(ii)  goal DIFF==0 α=6.0    : ' is_perfect_cube = is_perfect_cube ** 3 == abs(n)    \n    is_perfect_square = 
round(abs(n'

(iii) union DIFF==0 α=6.0   : ' is_perfect_square = int(abs(n) ** 0.5)    \n    is_perfect_square = 
is_perfect_square'

In [36]:
inst = build_reverse_example(random.Random(28), target_slot=_slot, n_cond=5)
test_prompt = inst["prompt"]
print(test_prompt)

test_tokenized = prepare_input(prompts=test_prompt, tokenizer=mt)

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given an integer n, checks the following conditions:

1. n is a perfect square
2. n is a perfect cube
3. n is a triangular number
4. n is smaller than the number obtained by reversing its digits
5. n is divisible by 16

For each condition, in the order listed, assign the boolean result of the check to its own local variable. After all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define any helper functions.
Do not include any comments in your 

In [37]:
for alph in [4.0, 6.0, 8.0, 12.0, 16]:
    diff_alpha = headpatched_generate(
        mt,
        dict(test_tokenized),
        DIFF0_GOAL,
        GOAL_MASK_,
        GOAL_LAYERS,
        n_steer_tokens=1,
        patch_input_positions=[-1, -2],
        alpha=alph,
    )
    rich_print(f"[cyan]DIFF==0 alpha={alph} :[/cyan] {escape(repr(diff_alpha))}")

DIFF==0 alpha=4.0 : ' is_perfect_cube = is_perfect_cube ** 3 == abs(n)    \n    is_perfect_square = round(abs(n) **
0.5)    \n    is_perfect_square = is_perfect_square ** 2 == abs(n)    \n    print(f"Perfect Square: 
{is_perfect_square'

DIFF==0 alpha=6.0 : ' is_perfect_cube = is_perfect_cube ** 3 == abs(n)    \n    is_perfect_square = round(abs(n) **
0.5)    \n    is_perfect_square = is_perfect_square ** 2 == abs(n)    \n    print(f"Perfect Square: 
{is_perfect_square'

DIFF==0 alpha=8.0 : ' is_perfect_cube = is_perfect_cube ** 3 == abs(n)    \n    is_perfect_square = round(abs(n) **
0.5)    \n    is_perfect_square = is_perfect_square ** 2 == abs(n)    \n    print(f"Perfect Square: 
{is_perfect_square'

DIFF==0 alpha=12.0 : ' d = round(abs(n) ** (1 / 3))    \n    is_perfect_cube = d ** 3 == abs(n)    \n    
is_perfect_square = int(abs(n) ** 0.5)    \n    is_perfect_square = is_perfect_square ** 2 == abs(n)'

DIFF==0 alpha=16 : ' d = round(abs(n) ** (1 / 3))    \n    is_perfect_cube = d ** 3 == abs(n)    \n    
is_perfect_square = int(abs(n) ** 0.5)    \n    is_perfect_square = is_perfect_square ** 2 == abs(n)'

## D5 · Move-on generalization — Spec-1 template (combined palindrome ∧ vowel filter)

Does the boundary move-on vector (cached on int-predicate multi-line prompts) still fire on a
**different template** — Spec-1 of `00_diff`? TASK = one combined filter (*palindromes AND starts with
a vowel*); the prefill implements **one** condition into `res` and sits at `return`. Clean, the model
picks up the **left-out** condition (`return [w for w in res if <other>]`). Fire move-on at the pivot →
does it instead just `return res` (moved on prematurely, treating the partial as done)?

In [19]:
# ── D5: Spec-1 template (exact) — partial filter in `res`, boundary move-on @ pivot ──
QUESTION_TMPL = ("Write a Python program that implements a function `func` which, given a list of "
                 "strings `words`, computes:\n<<task>>\n")
PREFILL_TMPL  = "<code>\ndef func(words: list[str]):\n    res = {progress}\n    return"
TASK_G = "return all the words in `words` that are palindromes and starts with a vowel"
PROG_G = "[word for word in words if word == word[::-1]]"          # only palindrome — vowel left out
PATCH_IDX = [-1, -2, -3]
g_prompt = get_prompt(QUESTION_TMPL.replace("<<task>>", TASK_G), PREFILL_TMPL.format(progress=PROG_G))
g_inp = prepare_input(prompts=g_prompt, tokenizer=mt)
rich_print(Panel(Text.from_markup(exact_prompt(g_prompt, patch_pos=PATCH_IDX)),
                 title="[bold]D5 DEST · Spec-1 template · pivot (last token) in salmon[/bold]",
                 title_align="left", border_style="grey42", padding=(1, 1)))
g_clean  = headpatched_generate(mt, dict(g_inp), MOVE_ON_DEC, get_mask_bin([], dec_layers), dec_layers,
                                max_new_tokens=24, patch_input_positions=PATCH_IDX)
g_moveon = headpatched_generate(mt, dict(g_inp), MOVE_ON_DEC, DEC_MASK, dec_layers, n_steer_tokens=1,
                                patch_input_positions=PATCH_IDX, alpha=ALPHA)
rich_print(f"[grey70]clean (expect: picks up vowel):[/grey70] {escape(repr(g_clean))}")
rich_print(f"[salmon1](i) move-on (expect: →return):[/salmon1] {escape(repr(g_moveon))}")

╭─ D5 DEST · Spec-1 template · pivot (last token) in salmon ──────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ return all the words in `words` that are palindromes and starts with a vowel\n                                  │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [word for word in words if word == word[::-1]]\n                                                      │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean (expect: picks up vowel): " [word for word in res if word[0].lower() in 'aeiou']\n</code>"

(i) move-on (expect: →return): " res\n\ndef func(words: list[str]):\n    vowels = {'a', 'e', 'i', 'o', 'u', 'A', 
'E', 'I', 'O', 'U'}\n    return [word for word in words if word == word[::-1] and word[0] in vowels"